# 📘 Notebook 01: Text as Data

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module A: Text as Data · Notebook 1 of 3 in this module · (1 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a string really is: a sequence of **characters**, each stored as a number
- Tell the difference between **characters** and **bytes**, and say what UTF-8 does
- Recognise text that *looks* identical but is not, and repair it with **Unicode normalisation**
- Turn a raw text into a list of words and **count** them
- Distinguish **tokens** from **types** and compute simple corpus statistics
- Observe **Zipf's law** in a real book and explain why it matters for everything that follows

> **Prerequisites:** Python basics: strings, loops, functions, lists and dictionaries.
>
> 🔭 **Why this notebook matters:** every method in this course, from a word counter to a large language model, begins by turning text into numbers. Before we do anything clever, we need to see what text actually looks like to a computer, and how unevenly words are distributed in real language. That unevenness shapes every design decision that comes later.

> 📦 **Libraries used in this notebook.**
>
> - **NLTK** (the Natural Language Toolkit) is the classic teaching library for NLP, developed since 2001 at the University of Pennsylvania. It offers tokenisers, taggers and parsers and, most usefully for us, dozens of ready-made text collections. We use it throughout the course. Here it supplies the text of a novel.
> - **Matplotlib** is the standard plotting library of Python, introduced in *From Python to Deep Learning & Fine-Tuning*, Notebook 07. We use it for one plot.
>
> The cell below installs what is needed. In Google Colab these libraries are already present and the cell finishes in a moment. On your own computer it may take a minute the first time.

In [ ]:
%pip install -q nltk matplotlib

## 1. Why language is hard for a computer

### Intuition first
Read this sentence: *"I saw her duck."* Did she lower her head, or does she own a bird? You decided in a fraction of a second, using the situation, your knowledge of the world and the sentences around it. A computer has none of that. It has a row of symbols.

Human language is **ambiguous** (one form, many meanings), **variable** (many forms, one meaning: *car*, *automobile*, *motor*), and **open-ended** (new words appear every day). Programming languages were designed to have none of these properties. Natural languages were not designed at all.

### What we are going to do about it
A computer can only calculate. So the whole field of **natural language processing (NLP)** is a long answer to one question:

> **How do we turn text into numbers without losing what it means?**

This notebook gives the first and simplest answer: count things. Let us start from the very bottom, with a single word.

## 2. A string is a sequence of characters

### Intuition first
To you, a word is a unit of meaning. To Python, a word is a row of boxes, each holding one character. You can ask how many boxes there are, look into any box by its position, and walk along the row.

In [ ]:
word = "language"

print(len(word))          # how many characters
print(word[0], word[-1])  # the first and the last character
print(word[0:4])          # a slice: characters 0, 1, 2, 3

for character in "cat":
    print(character)

### Formal definition
Each character is stored as a whole number called its **code point**. The table that assigns a number to every character of every writing system is called **Unicode**. Python gives you both directions:

- `ord(character)` returns the code point of a character
- `chr(number)` returns the character with that code point

In [ ]:
for character in "cat":
    print(character, ord(character))

print(chr(97), chr(98), chr(99))
print(chr(947), chr(955), chr(974), chr(963), chr(963), chr(945))   # a Greek word
print(chr(128512))                                                  # emoji are characters too

The six numbers on the second line spell *γλώσσα*, the Greek word for *language* (and for *tongue*: ambiguity is everywhere). Unicode has room for more than a million characters, which is why one Python string can mix English, Greek, Chinese and emoji without any special handling.

> 🧠 Notice that `"a"` is 97 and `"A"` is 65. To the computer, upper case and lower case are simply **different characters**. The idea that `"The"` and `"the"` are the same word is ours, not the machine's. We will have to tell it.

## 3. Characters are not bytes

### Intuition first
A code point is an abstract number. To save text in a file or send it over a network, that number must be written down as **bytes**. The rule for doing so is called an **encoding**. Think of the code point as a quantity and the encoding as the handwriting used to record it.

### Formal definition
**UTF-8** is the encoding used by almost all text on the web. It is a *variable-length* encoding: a character takes one byte if it is plain English, and two, three or four bytes otherwise. `text.encode("utf-8")` turns a string into bytes, and `data.decode("utf-8")` turns bytes back into a string.

In [ ]:
for text in ["cat", "γάτα", "猫", "😀"]:
    data = text.encode("utf-8")
    print(text, "| characters:", len(text), "| bytes:", len(data), "|", list(data))

The same animal costs 3 bytes in English, 8 in Greek and 3 in Chinese. The count of **characters** and the count of **bytes** are different things, and confusing them is a classic source of bugs.

What happens if we read bytes with the wrong encoding? We get nonsense, known as *mojibake*:

In [ ]:
data = "γλώσσα".encode("utf-8")

print(data.decode("utf-8"))      # the right encoding
print(data.decode("latin-1"))    # the wrong one: no error, just garbage

> ⚠️ **Common pitfalls**
>
> - Opening a file without stating its encoding. Always write `open(path, encoding="utf-8")`. The default differs between Windows, macOS and Linux, so a script that works on your laptop can produce garbage on another machine.
> - Assuming one character is one byte. It is true only for plain English text.
> - Seeing `Î³Î»ÏŽÏƒÏƒÎ±` in your data and deleting the "strange symbols". That is a decoding mistake made earlier. The fix is to decode correctly, not to throw characters away.

## 4. When equal text is not equal

### Intuition first
Here is a subtler trap. Some characters can be written in **two different ways** that look exactly the same on screen. The letter *é* exists as a single ready-made character, and also as a plain *e* followed by a separate "combining" accent that is drawn on top of it. Your eyes cannot tell them apart. Python can, and it says they are different.

In [ ]:
one = "café"      # é as one single character
two = "café"     # e followed by a combining acute accent

print(one, two)
print("Equal?", one == two)
print("Lengths:", len(one), len(two))

Two strings that print identically, with different lengths, that are not equal. If these were entries in a word count, *café* would be counted as two separate words.

### Formal definition
**Unicode normalisation** rewrites a string into one agreed form. The two forms you need to know are:

- **NFC** (composed): use the single ready-made character wherever one exists
- **NFD** (decomposed): split every accented character into base letter plus combining marks

Normalise all text to the same form once, as soon as you load it, and the problem disappears.

In [ ]:
import unicodedata

one_nfc = unicodedata.normalize("NFC", one)
two_nfc = unicodedata.normalize("NFC", two)

print("Equal after NFC?", one_nfc == two_nfc)
print("Lengths:", len(one_nfc), len(two_nfc))

The decomposed form is useful too. Once the accents are separate characters, we can simply drop them. That gives us accent-insensitive matching, which every search box needs.

In [ ]:
def strip_accents(text):
    decomposed = unicodedata.normalize("NFD", text)
    kept = ""
    for character in decomposed:
        if unicodedata.category(character) != "Mn":   # "Mn" is the category of combining marks
            kept += character
    return kept

print(strip_accents("café"))
print(strip_accents("γλώσσα"))
print(strip_accents("naïve résumé"))

### Upper and lower case
The other everyday source of "same word, different string" is case. `lower()` handles most of it, and it is smarter than it looks:

In [ ]:
print("The CAT".lower())
print("ΟΔΟΣ".lower())            # Greek has a special sigma at the end of a word
print("Straße".lower(), "|", "Straße".casefold())

Python wrote the final sigma of *οδός* correctly as `ς`. The last line shows `casefold()`, a more aggressive version of `lower()` meant for comparisons: it turns the German `ß` into `ss`.

> 🧠 Every one of these steps (normalising, removing accents, lower-casing) **throws information away** in order to treat different strings as the same word. That is a trade. *Apple* the company and *apple* the fruit become one word. *résumé* and *resume* become one word. Whether the trade is worth it depends on the task, and we will meet this decision again and again.

## 5. From a string to words

### Intuition first
A text is one long string. To count words we first have to cut the string into pieces. The obvious place to cut is at the spaces.

In [ ]:
sentence = "The cat sat on the mat. The dog did not."

words = sentence.split()
print(words)
print(len(words), "pieces")

Look closely at the result. Two things have gone wrong:

- `"mat."` and `"not."` have a full stop glued to them, so `"mat."` would not match `"mat"`.
- `"The"` and `"the"` are different strings.

Let us write a slightly better splitter by hand: lower-case everything, and keep only letters. Every other character becomes a space.

In [ ]:
def simple_words(text):
    cleaned = ""
    for character in text.lower():
        if character.isalpha() or character == "'":
            cleaned += character          # keep letters and apostrophes
        else:
            cleaned += " "                # everything else separates words

    words = []
    for piece in cleaned.split():
        piece = piece.strip("'")          # remove quotation marks at the edges
        if piece != "":
            words.append(piece)
    return words

print(simple_words(sentence))
print(simple_words("Don't panic: it's only 'text'!"))

This is our first **tokeniser**: a function that cuts text into units. It keeps the apostrophe inside *don't* and removes the quotation marks around *text*. It is also far from perfect. It destroys numbers, web addresses and hyphenated words, and it has no idea what to do with a language that does not use spaces. Notebook 03 is devoted to doing this properly. For now it is good enough to start counting.

## 6. Counting words

### Intuition first
Ask someone what a book is about without reading it. A reasonable trick is to look at which words appear most often. Counting is the oldest method in NLP, and it remains the foundation of many newer ones.

### Formal definition
Two terms are used constantly from here on:

- A **token** is one occurrence of a word in the text. *"the cat and the dog"* has **5 tokens**.
- A **type** is a distinct word. The same sentence has **4 types**, because *the* occurs twice.

The set of all types in a text is its **vocabulary**. Let us count by hand first, with a dictionary.

In [ ]:
words = simple_words("the cat and the dog and the bird")

counts = {}
for word in words:
    if word in counts:
        counts[word] += 1
    else:
        counts[word] = 1

print(counts)
print("Tokens:", len(words))
print("Types: ", len(counts))

Python has this pattern built in. `Counter` from the `collections` module does the counting in one line, and adds a very convenient `most_common` method.

In [ ]:
from collections import Counter

counts = Counter(words)

print(counts)
print(counts.most_common(2))
print(counts["the"], counts["elephant"])    # a word that never appeared has count 0

## 7. A real book

Toy sentences hide the interesting behaviour. Let us load a complete book: *Alice's Adventures in Wonderland* by Lewis Carroll, which is in the public domain. It comes with **NLTK**, the Natural Language Toolkit, a Python library we will use throughout the course. The first line downloads a small collection of classic books.

In [ ]:
import nltk
nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg

alice = gutenberg.raw("carroll-alice.txt")

print(len(alice), "characters")
print()
print(alice[:400])

A collection of text prepared for study is called a **corpus** (plural: *corpora*). Ours is one book. Let us measure it.

In [ ]:
alice_words = simple_words(alice)
alice_counts = Counter(alice_words)

tokens = len(alice_words)
types = len(alice_counts)

print(f"Tokens: {tokens:,}")
print(f"Types:  {types:,}")
print(f"On average, each word is used {tokens / types:.1f} times.")

In [ ]:
print(f"{'rank':>4}  {'word':<8} {'count':>6}")
for rank, (word, count) in enumerate(alice_counts.most_common(15), start=1):
    print(f"{rank:>4}  {word:<8} {count:>6}")

The list is disappointing if we hoped to learn what the book is about. Apart from *alice* and *said*, the top words are *the*, *and*, *to*, *a*, *she*, *it*, *of*: the small grammatical words that appear in every English text. They are called **function words**, and they tell us almost nothing about the topic.

Now look at the other end of the list.

In [ ]:
once = []
for word, count in alice_counts.items():
    if count == 1:
        once.append(word)

print(f"{len(once):,} of the {types:,} types appear exactly once.")
print(f"That is {len(once) / types:.0%} of the vocabulary.")
print(once[:25])

Nearly half the vocabulary of the book appears **only once**. A word that occurs a single time in a corpus has a name: a *hapax legomenon*. These rare words are where the content is (*marmalade*, *daisies*, *tunnel*), and there are a great many of them.

So a real text has a handful of words used all the time and a long list of words used almost never. This is not a peculiarity of Lewis Carroll.

## 8. Zipf's law

### Intuition first
Sort the words of any large text from most frequent to least frequent and number them: rank 1, rank 2, rank 3 and so on. A remarkable regularity appears. The second word is used about half as often as the first, the third about a third as often, the tenth about a tenth as often.

### Formal definition
**Zipf's law** says that the frequency of a word is roughly inversely proportional to its rank:

$$\text{frequency} \approx \frac{C}{\text{rank}}$$

where $C$ is a constant for the text. If that is true, then **rank × frequency** should stay roughly the same all the way down the list. We can check.

In [ ]:
ranked = alice_counts.most_common()

print(f"{'rank':>5}  {'word':<10} {'count':>6} {'rank x count':>13}")
for rank in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]:
    word, count = ranked[rank - 1]
    print(f"{rank:>5}  {word:<10} {count:>6} {rank * count:>13,}")

The count falls from over a thousand to single digits, a drop of several hundred times, while the product `rank x count` stays within the same order of magnitude. It is not an exact law, but it is a strikingly good approximation for something as untidy as a novel.

A relationship of the form $y = C / x$ becomes a **straight line** when both axes are drawn on a logarithmic scale. Let us plot every word of the book.

In [ ]:
import matplotlib.pyplot as plt

ranks = list(range(1, len(ranked) + 1))
frequencies = [count for word, count in ranked]
ideal = [frequencies[0] / rank for rank in ranks]

plt.figure(figsize=(7, 4))
plt.loglog(ranks, frequencies, label="Alice in Wonderland")
plt.loglog(ranks, ideal, linestyle="--", label="Zipf's law: C / rank")
plt.title("Word frequency against rank")
plt.xlabel("Rank of the word (log scale)")
plt.ylabel("Number of occurrences (log scale)")
plt.legend()
plt.grid(True)
plt.show()

The measured curve follows the straight dashed line closely across the whole vocabulary. The flat steps at the bottom right are the many words that occur exactly 3 times, exactly 2 times and exactly once.

### What this means in practice
One more measurement makes the consequence concrete: how much of the text is covered by the most frequent words?

In [ ]:
print(f"{'top types':>10} {'share of all tokens':>21}")
for top in [10, 50, 100, 500, 1000]:
    covered = 0
    for word, count in ranked[:top]:
        covered += count
    print(f"{top:>10} {covered / tokens:>20.0%}")

A hundred words, out of a vocabulary of thousands, account for more than half of everything written in the book. This has two consequences that will follow us through the entire course:

1. **The frequent words are few and carry little meaning.** Any method based on counting must stop them from drowning out everything else. Notebook 03 removes them, and Notebook 08 learns to down-weight them.
2. **Most words are rare, however much text you collect.** There will always be words your system has seen once, or never. Handling the unseen is the central difficulty of language modelling (Notebook 07), and the reason modern systems split words into smaller pieces (Notebook 15).

> 🧠 Zipf's law holds for English, Greek, Chinese, and every other human language tested, for novels, newspapers and chat messages alike. Whenever you design or evaluate an NLP system, assume your data looks like this.

## 9. What counting cannot see

Counting words gave us real information from a few lines of code. It is worth being honest about what it throws away. Consider two sentences.

In [ ]:
first = simple_words("the dog bit the man")
second = simple_words("the man bit the dog")

print(Counter(first))
print(Counter(second))
print("Same counts?", Counter(first) == Counter(second))

The two sentences describe opposite events and have exactly the same word counts. Counting ignores **order**. It also treats *dog* and *puppy* as unrelated, because they are different strings, and it treats the *bank* of a river and the *bank* that holds your money as one and the same.

Here is our first answer to the question of the course, with its score:

| | Word counts |
|---|---|
| Turns text into numbers | yes |
| Keeps the topic | partly |
| Keeps word order | no |
| Knows that two words are similar | no |
| Knows that one word has several meanings | no |

Every module that follows repairs one of those rows.

---
## ✏️ Exercises

### Exercise 1 (Code points)
Print each character of the word `"Ελλάδα"` together with its code point. Then build the string `"NLP"` using only `chr()` and the numbers 78, 76 and 80.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for character in "Ελλάδα":
    print(character, ord(character))

print(chr(78) + chr(76) + chr(80))
```

*The two letters λ have the same code point: the same character is always the same number.*
</details>

### Exercise 2 (Characters and bytes)
Write a function `describe(text)` that prints the number of characters and the number of UTF-8 bytes of a string. Try it on `"hello"`, `"γειά"` and `"hello 👋"`. Which characters cost more than one byte?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def describe(text):
    print(text, "| characters:", len(text), "| bytes:", len(text.encode("utf-8")))

describe("hello")
describe("γειά")
describe("hello 👋")
```

*English letters and the space take one byte each, Greek letters take two, and the emoji takes four.*
</details>

### Exercise 3 (The same word?)
Write a function `same_word(a, b)` that returns `True` when two strings are the same word if we ignore **case** and **accents**. It should say `True` for `"Café"` and `"cafe"`, and for `"ΓΛΏΣΣΑ"` and `"γλωσσα"`, and `False` for `"cat"` and `"cut"`. You may use `strip_accents` from Section 4.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def same_word(a, b):
    return strip_accents(a).casefold() == strip_accents(b).casefold()

print(same_word("Café", "cafe"))
print(same_word("ΓΛΏΣΣΑ", "γλωσσα"))
print(same_word("cat", "cut"))
```

*This is exactly what a search box does before it compares your query with its index.*
</details>

### Exercise 4 (The most common letters)
Count the **letters** of *Alice in Wonderland* (not the words) and print the eight most common. Ignore case, and ignore anything that is not a letter.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
letters = []
for character in alice.lower():
    if character.isalpha():
        letters.append(character)

letter_counts = Counter(letters)
for letter, count in letter_counts.most_common(8):
    print(letter, count, f"{count / len(letters):.1%}")
```

*The letter e alone is about one letter in eight. Uneven letter frequencies are what made simple substitution ciphers breakable centuries before computers existed.*
</details>

### Exercise 5 (The longest words)
Print the ten longest **distinct** words in the book, longest first. (Hint: `sorted(collection, key=len, reverse=True)`.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
longest = sorted(alice_counts, key=len, reverse=True)
for word in longest[:10]:
    print(len(word), word, alice_counts[word])
```

*Looping over a `Counter` gives its keys, which are the types. Notice how rare the long words are: frequent words tend to be short. That observation is also due to Zipf.*
</details>

### Exercise 6 (Vocabulary growth)
The **type-token ratio** of a text is `types / tokens`. Compute it for the first 1,000 words of the book, the first 5,000, the first 10,000, and the whole book. What happens to the ratio as the text grows, and why?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for size in [1000, 5000, 10000, len(alice_words)]:
    part = alice_words[:size]
    ratio = len(set(part)) / len(part)
    print(f"{size:>7,} tokens  {len(set(part)):>6,} types  ratio {ratio:.2f}")
```

*The ratio falls steadily. In the first thousand words, more than one word in three is new. Later, most words have been seen already, but new ones never stop arriving. For this reason the type-token ratios of two texts can only be compared when the texts have the same length.*
</details>

### Exercise 7 (Another book, a little harder)
Load Jane Austen's *Emma* with `gutenberg.raw("austen-emma.txt")`. For both books, print the number of tokens, the number of types, and the share of types that occur only once. Then print the 15 most common words of *Emma* next to those of *Alice*. What do the two lists have in common, and where do they differ?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
emma_words = simple_words(gutenberg.raw("austen-emma.txt"))
emma_counts = Counter(emma_words)

for title, counts in [("Alice", alice_counts), ("Emma", emma_counts)]:
    total = sum(counts.values())
    seen_once = 0
    for count in counts.values():
        if count == 1:
            seen_once += 1
    print(f"{title:<6} tokens {total:>8,}  types {len(counts):>6,}  seen once {seen_once / len(counts):.0%}")

print()
pairs = zip(alice_counts.most_common(15), emma_counts.most_common(15))
for (word_a, count_a), (word_e, count_e) in pairs:
    print(f"{word_a:<8} {count_a:>5}     {word_e:<8} {count_e:>5}")
```

*Emma is about six times longer, yet its vocabulary is less than three times larger: vocabulary grows much more slowly than text. The two top-15 lists are almost the same function words. The differences that remain are telling: Alice has "alice" and "said", Emma has "her", "he" and "not". Even at the very top of the list, a trace of the content shows through.*
</details>

## 🔑 Key takeaways

- A string is a sequence of **characters**, and each character is a number, its Unicode **code point**.
- Characters are stored as **bytes** through an encoding. Use **UTF-8**, and state it explicitly when opening files.
- Text that looks the same can differ: **normalise** (NFC), and decide deliberately about case and accents.
- Cutting text into units is **tokenisation**. Splitting on spaces is a start, not a solution.
- A **token** is an occurrence, a **type** is a distinct word, and the set of types is the **vocabulary**.
- **Zipf's law**: a few words are extremely frequent and most words are rare. About half the vocabulary of a book appears once.
- Word counts turn text into numbers, but they lose **order**, **similarity** and **word sense**.

---
**Next:** *Notebook 02: Regular Expressions*, a compact language for describing patterns in text, which will let us clean and cut text far more precisely than `simple_words` can.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*